In [10]:
# =============================================================
# Cargar datos, preprocesamiento, modelos y top3 del Sprint 3
# =============================================================

import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

# IMPORTANTE: Pipeline correcto para sampling Y para preprocesamiento
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import RandomOverSampler

# Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier,
    ExtraTreesClassifier
)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

# =============================================================
# 1. Cargar conjunto de datos
# =============================================================
df = pd.read_csv("../data/processed/04_default_credit_features.csv")
df = df.drop(columns=["ID"])

TARGET = "default payment next month"
X = df.drop(columns=[TARGET])
y = df[TARGET]

# Columnas
cat_cols = X.select_dtypes(include=["object", "category", "string"]).columns.tolist()
num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

# =============================================================
# 2. Preprocesamiento (usando imblearn.Pipeline)
# =============================================================
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preproc = ColumnTransformer([
    ("num", numeric_pipeline, num_cols),
    ("cat", categorical_pipeline, cat_cols)
])

# =============================================================
# 3. Modelos optimizados (Sprint 3)
# =============================================================
modelos = {
    "LogisticRegression": LogisticRegression(max_iter=2000),
    "DecisionTree": DecisionTreeClassifier(max_depth=5),
    "RandomForest": RandomForestClassifier(n_estimators=20, n_jobs=-1),
    "SVM": SVC(probability=True),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "GradientBoosting": GradientBoostingClassifier(),
    "AdaBoost": AdaBoostClassifier(),
    "ExtraTrees": ExtraTreesClassifier(n_estimators=50, n_jobs=-1)
}

# =============================================================
# 4. Balanceo optimizado
# =============================================================
estrategias = {
    "NoBalance": None,
    "Over": RandomOverSampler(random_state=42)
}

# =============================================================
# 5. Validación
# =============================================================
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
metricas = ["accuracy", "precision", "recall", "f1", "roc_auc"]
resultados = {}

# =============================================================
# 6. Entrenamiento (Sprint 3)
# =============================================================
for nombre_modelo, modelo in modelos.items():
    for nombre_estrategia, sampler in estrategias.items():

        steps = [("preprocessing", preproc)]

        if sampler is not None:
            steps.append(("sampling", sampler))

        steps.append(("clf", modelo))

        # Pipeline CORRECTO (imblearn)
        pipe = Pipeline(steps)

        puntuaciones = cross_validate(
            pipe, X, y, cv=cv, scoring=metricas, n_jobs=-1
        )

        resultados[f"{nombre_modelo}_{nombre_estrategia}"] = {
            m: puntuaciones[f"test_{m}"].mean() for m in metricas
        }

# =============================================================
# 7. Ranking y Top 3
# =============================================================
tabla = pd.DataFrame(resultados).T
ranking = tabla.sort_values(by="f1", ascending=False)
top3 = ranking.head(3)

print("=== Top 3 modelos ===")
display(top3)

=== Top 3 modelos ===


,accuracy,precision,recall,f1,roc_auc
SVM_Over,0.773933,0.491030,0.597498,0.539021,0.765311
GradientBoosting_Over,0.762700,0.472427,0.622815,0.537288,0.780309
AdaBoost_Over,0.766300,0.477477,0.595690,0.530030,0.770839


In [12]:
import os
import joblib
import optuna
import pandas as pd

from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    StratifiedKFold,
    cross_validate
)

from imblearn.pipeline import Pipeline
from imblearn.over_sampling import RandomOverSampler

print("=== Sprint 4: Hyperparameter Tuning (Optimizado) ===")

# =============================================================
# 1. Espacios de búsqueda reducidos (rápidos pero efectivos)
# =============================================================
param_spaces = {
    "SVM_Over": {
        "clf__C": [0.1, 1],
        "clf__kernel": ["linear", "rbf"],
        "clf__gamma": ["scale"]
    },
    "GradientBoosting_Over": {
        "clf__n_estimators": [50, 100],
        "clf__learning_rate": [0.05, 0.1],
        "clf__max_depth": [2, 3]
    },
    "AdaBoost_Over": {
        "clf__n_estimators": [50, 100],
        "clf__learning_rate": [0.05, 0.1]
    }
}

# =============================================================
# 2. CV estratificado más rápido (2 folds)
# =============================================================
cv = StratifiedKFold(n_splits=2, shuffle=True, random_state=42)

# =============================================================
# 3. Tuning rápido: Grid + Random + Optuna (5 trials)
# =============================================================
tuning_results = {}
os.makedirs("models/tuned", exist_ok=True)

for modelo_nombre in top3.index:

    print(f"\n--- Tuning para {modelo_nombre} ---")

    modelo_base = modelos[modelo_nombre.split("_")[0]]
    grid = param_spaces.get(modelo_nombre, None)

    if grid is None:
        print(f"No hay grid definido para {modelo_nombre}, se omite.")
        continue

    pipe = Pipeline([
        ("preprocessing", preproc),
        ("sampling", RandomOverSampler(random_state=42)),
        ("clf", modelo_base)
    ])

    # ---------------- GRID SEARCH ----------------
    grid_search = GridSearchCV(
        estimator=pipe,
        param_grid=grid,
        scoring="f1",
        cv=cv,
        n_jobs=-1
    )
    grid_search.fit(X, y)

    # ---------------- RANDOM SEARCH ----------------
    random_search = RandomizedSearchCV(
        estimator=pipe,
        param_distributions=grid,
        n_iter=5,
        scoring="f1",
        cv=cv,
        n_jobs=-1,
        random_state=42
    )
    random_search.fit(X, y)

=== Sprint 4: Hyperparameter Tuning (Optimizado) ===

--- Tuning para SVM_Over ---


c:\Users\Elenita\miniforge3\envs\proyecto1\Lib\site-packages\sklearn\model_selection\_search.py:324: UserWarning: The total space of parameters 4 is smaller than n_iter=5. Running 4 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(



--- Tuning para GradientBoosting_Over ---

--- Tuning para AdaBoost_Over ---


c:\Users\Elenita\miniforge3\envs\proyecto1\Lib\site-packages\sklearn\model_selection\_search.py:324: UserWarning: The total space of parameters 4 is smaller than n_iter=5. Running 4 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


In [14]:
from sklearn.metrics import f1_score

# ---------------- OPTUNA (5 trials) ----------------
def objective(trial):
    params = {}
    for key, values in grid.items():
        if isinstance(values[0], float):
            params[key] = trial.suggest_float(key, min(values), max(values))
        elif isinstance(values[0], int):
            params[key] = trial.suggest_int(key, min(values), max(values))
        else:
            params[key] = trial.suggest_categorical(key, values)

    pipe_opt = Pipeline([
        ("preprocessing", preproc),
        ("sampling", RandomOverSampler(random_state=42)),
        ("clf", modelo_base.set_params(**{
            k.replace("clf__", ""): v for k, v in params.items()
        }))
    ])

    # Entrenamos y predecimos manualmente para calcular F1
    pipe_opt.fit(X, y)
    y_pred = pipe_opt.predict(X)

    # F1 macro calculado manualmente (siempre funciona)
    return f1_score(y, y_pred, average="macro")

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=5)

[I 2026-05-05 21:51:33,926] A new study created in memory with name: no-name-7019c212-dd06-48d5-a176-bd1b0b221413
[I 2026-05-05 21:51:42,060] Trial 0 finished with value: 0.6840138021902513 and parameters: {'clf__n_estimators': 98, 'clf__learning_rate': 0.0630882180929879}. Best is trial 0 with value: 0.6840138021902513.
[I 2026-05-05 21:51:48,624] Trial 1 finished with value: 0.6840138021902513 and parameters: {'clf__n_estimators': 80, 'clf__learning_rate': 0.06390810681146539}. Best is trial 0 with value: 0.6840138021902513.
[I 2026-05-05 21:51:53,070] Trial 2 finished with value: 0.6840138021902513 and parameters: {'clf__n_estimators': 54, 'clf__learning_rate': 0.08479102080046133}. Best is trial 0 with value: 0.6840138021902513.
[I 2026-05-05 21:51:59,341] Trial 3 finished with value: 0.6840138021902513 and parameters: {'clf__n_estimators': 76, 'clf__learning_rate': 0.06964298690474276}. Best is trial 0 with value: 0.6840138021902513.
[I 2026-05-05 21:52:04,585] Trial 4 finished wi

In [15]:
# 4. Comparación before vs after
# =============================================================
before = top3.loc[modelo_nombre]["f1"]
after = max(grid_search.best_score_, random_search.best_score_, study.best_value)

# =============================================================
# 5. Guardar modelo tuneado
# =============================================================
best_params = study.best_params
best_model = modelo_base.set_params(**{k.replace("clf__", ""): v for k, v in best_params.items()})

final_pipe = Pipeline([
        ("preprocessing", preproc),
        ("sampling", RandomOverSampler(random_state=42)),
        ("clf", best_model)
    ])

final_pipe.fit(X, y)
joblib.dump(final_pipe, f"models/tuned/{modelo_nombre}_tuned.pkl")

# =============================================================
# 6. Documentar resultados
# =============================================================
tuning_results[modelo_nombre] = {
        "before_f1": before,
        "after_f1": after,
        "best_params_grid": grid_search.best_params_,
        "best_params_random": random_search.best_params_,
        "best_params_optuna": study.best_params
    }

# =============================================================
# Tabla final de resultados
# =============================================================
tuning_df = pd.DataFrame(tuning_results).T
print("\n=== Resultados Finales Sprint 4 (Optimizado) ===")
display(tuning_df)



=== Resultados Finales Sprint 4 (Optimizado) ===


,before_f1,after_f1,best_params_grid,best_params_random,best_params_optuna
AdaBoost_Over,0.53003,0.684014,"{'clf__learning_rate': 0.1, 'clf__n_estimators...","{'clf__n_estimators': 100, 'clf__learning_rate...","{'clf__n_estimators': 98, 'clf__learning_rate'..."
